In [2]:
import sys, os
import warnings
from tqdm import TqdmWarning
warnings.filterwarnings("ignore", category=TqdmWarning)
# if os.path.abspath(os.path.join(os.getcwd(), "..")) not in sys.path:
#     sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from scalaremlp.nn.objax import InvarianceLayer_objax
from trainer.hamiltonian_dynamics import IntegratedDynamicsTrainer, DoubleSpringPendulum, hnnScalars_trial
from torch.utils.data import DataLoader
from oil.utils.utils import FixedNumpySeed, FixedPytorchSeed
from trainer.utils import LoaderTo
from oil.datasetup.datasets import split_dataset
from oil.tuning.args import argupdated_config
import torch.nn as nn
import logging
import scalaremlp
import scalaremlp.reps
import objax
import itertools
from hnn_scalars import makeTrainerScalars

# print(sys.argv)

## hnn scalars

In [ ]:
layer_num_li = [3, 5, 7]
hidden_layer_num_li = [100, 150, 200]
lr_li = [1e-2, 5e-3, 3e-3]

levels = {'critical': logging.CRITICAL, 'error': logging.ERROR,
          'warn': logging.WARNING, 'warning': logging.WARNING,
          'info': logging.INFO, 'debug': logging.DEBUG}

if __name__ == '__main__':
    if len(sys.argv)!=1:
        sys.argv=[sys.argv[0]]  # Remove IPython kernel argument
    parameter_comb = list(itertools.product(layer_num_li, hidden_layer_num_li, lr_li))
    # parameter_comb = [
    #     (5,100,0.01)
    # ]
    
    already_run=[
        # (3, 100, 0.01), (3, 100, 5e-3), (3, 100, 3e-3),
        # (3, 150, 0.01)
    ]
    with open(file="parameter_tune_result.txt", mode="a") as f:
        f.write("hnnScalars parameter tuning results\n")
        f.write("="*50+"\n")
    for parameter in parameter_comb:
        layer_num, hidden_layer_num, lr = parameter
        if parameter in already_run:
            print(f"Skipping already run parameters: layers={layer_num}, hidden_layers={hidden_layer_num}, lr={lr}")
            continue
        print(f"Running with parameters: layers={layer_num}, hidden_layers={hidden_layer_num}, lr={lr}")
        Trial = hnnScalars_trial(makeTrainerScalars)

        # change the parameter setting
        makeTrainerScalars.__kwdefaults__["net_config"]["n_layers"] = layer_num
        makeTrainerScalars.__kwdefaults__["net_config"]["n_hidden"] = hidden_layer_num
        makeTrainerScalars.__kwdefaults__["lr"] = lr
        makeTrainerScalars.__kwdefaults__["num_epochs"]=1
        makeTrainerScalars.__kwdefaults__["save"]=False
        
        kwds = makeTrainerScalars.__kwdefaults__.copy()
        # 1
        cfg, outcome = Trial(argupdated_config(makeTrainerScalars.__kwdefaults__))
        
        # 2
        # print("makeTrainerScalars.__kwdefaults__:", makeTrainerScalars.__kwdefaults__)
        # cfg, outcome = Trial(kwds)

        # write the result to a file
        current_mode="w" if parameter_comb.index(parameter)==0 else "a"
        with open(file="parameter_tune_result.txt", mode=current_mode) as f:
            parameter_w="-".join(map(str, parameter))
            f.write(f"layer number-hidden layer number-learning rate: {parameter_w}\n")
            outcome_str = [s.strip() for s in str(outcome).split("\n")]
            f.write(f"{outcome_str[0]}: {"-".join(outcome_str[1].split()[1:])}\n")
            f.write("-"*50)
            f.write("\n") 


Running with parameters: layers=3, hidden_layers=100, lr=0.01


train:   0%|          | 0/1 [00:00<?, ?it/s]

## neural ODE scalars